# 01 — Extract WQP data

This notebook pulls California WQP results for:
- pH
- Turbidity
- Dissolved oxygen

Goal: build the discrete water-quality portion of the modeling dataset.

In [ ]:
import sys
from pathlib import Path
import yaml
import pandas as pd

project_root = Path("..").resolve()
sys.path.append(str(project_root / "src"))

from wqp import pull_characteristics_with_synonyms, aggregate_wqp_daily, add_usgs_site_no

In [ ]:
cfg = yaml.safe_load(open(project_root / "config" / "config.sample.yaml"))
cfg["project"]

In [ ]:
wqp_long = pull_characteristics_with_synonyms(
    statecode=cfg["project"]["state_code"],
    start_date=cfg["project"]["start_date"],
    end_date=cfg["project"]["end_date"],
    site_type=cfg["project"]["site_type"],
    characteristics=cfg["wqp"]["characteristics"],
    synonym_map=cfg["wqp"]["characteristic_synonyms"],
)
wqp_long.head()

In [ ]:
wqp_wide = aggregate_wqp_daily(wqp_long)
wqp_wide = add_usgs_site_no(wqp_wide)
wqp_wide.head()

In [ ]:
print("Long rows:", len(wqp_long))
print("Wide rows:", len(wqp_wide))
print("USGS-backed sites:", wqp_wide["usgs_site_no"].notna().sum())

In [ ]:
(project_root / "data" / "interim").mkdir(parents=True, exist_ok=True)
wqp_long.to_csv(project_root / "data" / "raw" / "wqp_long.csv", index=False)
wqp_wide.to_csv(project_root / "data" / "interim" / "wqp_wide.csv", index=False)